# Explainability — Exp 3 Best (SwinV2-Small, Two-Phase)

**EEEM068 Applied Machine Learning — Group Coursework**

| Section | Technique |
|---|---|
| 1 | Setup — install, load checkpoint, collect predictions |
| 2 | Per-grade sample images (Kaggle-style visual) |
| 3 | Grad-CAM++ — spatial heatmaps |
| 4 | Per-grade Grad-CAM gallery |
| 5 | Attention Rollout — native SwinV2 attention |
| 6 | Grade 1 false negatives — clinically critical |
| 7 | LIME — superpixel explanations |
| 8 | Misclassification analysis |
| 9 | Confidence calibration |
| 10 | Summary |

## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))
from utils import *
print('utils loaded ✓')

In [ ]:
# ── Install explainability dependencies into the active venv ──────────────────
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'grad-cam', 'lime', 'scikit-image', '--quiet'], check=True)
print('Dependencies installed ✓')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    confusion_matrix, cohen_kappa_score,
    f1_score, precision_score, recall_score,
    accuracy_score,
)
from sklearn.calibration import calibration_curve

# ── Paths (mirror the training notebook exactly) ──────────────────────────────
EXP_DIR   = ROOT / 'outputs' / 'swinv2_small_exp3'
CKPT_PATH = EXP_DIR / 'checkpoints' / 'exp3_best.pt'
XAI_DIR   = EXP_DIR / 'explainability'
XAI_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
DR_COLORS   = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']

print(f'Checkpoint : {CKPT_PATH}')
print(f'XAI output : {XAI_DIR}')

In [ ]:
# ── Reload model from best checkpoint ────────────────────────────────────────
set_seed()
model     = build_swin_model(pretrained=False).to(device)
criterion = FocalLoss(alpha=None, gamma=2.0)
scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded epoch={ckpt["epoch"]}  val_kappa={ckpt["val_kappa"]:.4f}')

# ── Test dataset & loader ─────────────────────────────────────────────────────
test_ds     = DRDataset(test_df, TEST_LABELS_DIR, transform=val_transform)
test_loader = make_val_loader(test_ds)
print(f'Test set: {len(test_ds):,} images')

In [ ]:
# ── Collect all predictions in one forward pass ───────────────────────────────
# All downstream sections use these arrays — no repeated forward passes needed
all_probs, all_preds, all_labels = [], [], []

model.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs  = imgs.to(device)
        probs = F.softmax(model(imgs), dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(probs.argmax(axis=1))
        all_labels.append(labels.numpy())

all_probs  = np.concatenate(all_probs,  axis=0)   # (N, 5)
all_preds  = np.concatenate(all_preds,  axis=0)   # (N,)
all_labels = np.concatenate(all_labels, axis=0)   # (N,)

print(f'Samples  : {len(all_preds):,}')
print(f'Accuracy : {(all_preds == all_labels).mean():.3f}')
print(f'QWK      : {cohen_kappa_score(all_labels, all_preds, weights="quadratic"):.4f}')

In [ ]:
# ── Helper utilities ──────────────────────────────────────────────────────────
def get_tensor(ds, idx):
    """Return ((1,C,H,W) tensor on device, int label)."""
    img, label = ds[idx]
    return img.unsqueeze(0).to(device), int(label)


def get_pil(ds, idx):
    """Return (PIL Image, int label) by un-normalising the stored tensor.
    No dependency on df column names — works regardless of dataset implementation.
    """
    tensor, label = ds[idx]
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img  = (tensor.cpu() * std + mean).clamp(0, 1)
    return TF.to_pil_image(img), int(label)


def to_np(pil_or_tensor, size=224):
    """PIL or (C,H,W) tensor -> float32 numpy (H,W,3) in [0,1]."""
    if isinstance(pil_or_tensor, torch.Tensor):
        arr = pil_or_tensor.cpu().numpy().transpose(1, 2, 0)
    else:
        arr = np.array(pil_or_tensor.resize((size, size))) / 255.0
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    return arr.astype(np.float32)


# Sanity check
pil, lbl = get_pil(test_ds, 0)
print(f'get_pil OK — label={lbl}, size={pil.size}')
print('Helpers defined ✓')

---
## 2. Per-Grade Sample Images

Mirrors the Kaggle winner's opening section: show what each DR severity grade looks like on a real retina, using the model's most confident correct test predictions. Sets clinical context before any explainability.

In [ ]:
N_SAMPLES = 3
fig, axes = plt.subplots(5, N_SAMPLES, figsize=(N_SAMPLES * 4, 5 * 3.5))
fig.suptitle('Test Set — Most Confident Correct Predictions per DR Grade',
             fontsize=14, fontweight='bold', y=1.01)

for grade in range(5):
    mask    = (all_labels == grade) & (all_preds == grade)
    indices = np.where(mask)[0]
    if len(indices) == 0:
        print(f'Grade {grade}: no correct predictions found.')
        for col in range(N_SAMPLES):
            axes[grade, col].axis('off')
        continue
    top_idx = indices[np.argsort(all_probs[indices, grade])[::-1][:N_SAMPLES]]

    for col, idx in enumerate(top_idx):
        ax = axes[grade, col]
        pil, _ = get_pil(test_ds, int(idx))
        ax.imshow(pil)
        ax.axis('off')
        conf = all_probs[idx, grade]
        if col == 0:
            ax.set_title(
                f'Grade {grade} — {CLASS_NAMES[grade]}\nConf: {conf*100:.1f}%',
                fontsize=10, fontweight='bold', color=DR_COLORS[grade], loc='left')
        else:
            ax.set_title(f'Conf: {conf*100:.1f}%', fontsize=9)

plt.tight_layout()
fig.savefig(XAI_DIR / 'per_grade_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {XAI_DIR / "per_grade_samples.png"}')

---
## 3. Grad-CAM++ — Spatial Heatmaps

Highlights which retinal regions drove each prediction.
SwinV2 outputs patch tokens `(B, N, C)` — the reshape transform converts these to a spatial grid `(B, C, H, W)` for Grad-CAM. The transform handles both shapes so it works regardless of which norm layer is targeted.

In [ ]:
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


def swin_reshape_transform(tensor):
    """Handle both SwinV2 output shapes:
    - (B, N, C): token sequence -> reshape to (B, C, H, W)
    - (B, C, H, W): already spatial -> pass through unchanged
    """
    if tensor.ndim == 4:
        return tensor
    B, N, C = tensor.shape
    H = W = int(N ** 0.5)
    assert H * W == N, f'Non-square patch grid: N={N}'
    return tensor.reshape(B, H, W, C).permute(0, 3, 1, 2)


# Initialise ONCE globally — never inside a plot function (causes hook accumulation)
cam = GradCAMPlusPlus(
    model=model,
    target_layers=[model.norm],
    reshape_transform=swin_reshape_transform,
)
print('Grad-CAM++ initialised on model.norm ✓')

In [ ]:
def gradcam_plot(ds, idx, save_path=None):
    """Plot: original | Grad-CAM++ overlay | class probability bars."""
    tensor, label = get_tensor(ds, idx)
    rgb_np        = to_np(get_pil(ds, idx)[0])

    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1).cpu().numpy()[0]
    pred = int(probs.argmax())

    heatmap = cam(input_tensor=tensor,
                  targets=[ClassifierOutputTarget(pred)])[0]
    overlay = show_cam_on_image(rgb_np, heatmap, use_rgb=True)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(
        f'Grad-CAM++  |  True: {CLASS_NAMES[label]}  '
        f'Pred: {CLASS_NAMES[pred]}  Conf: {probs[pred]*100:.1f}%',
        fontsize=12, fontweight='bold',
        color='green' if pred == label else 'red'
    )
    axes[0].imshow(rgb_np);   axes[0].set_title('Original');          axes[0].axis('off')
    axes[1].imshow(overlay);  axes[1].set_title('Grad-CAM++ Overlay'); axes[1].axis('off')

    bars = axes[2].bar(range(5), probs, color=DR_COLORS, edgecolor='white', width=0.6)
    axes[2].set_xticks(range(5))
    axes[2].set_xticklabels([f'G{i}' for i in range(5)])
    axes[2].set_ylabel('Probability'); axes[2].set_ylim(0, 1)
    axes[2].set_title('Class Probabilities')
    for bar, p in zip(bars, probs):
        axes[2].text(bar.get_x() + bar.get_width()/2, p + 0.02,
                     f'{p:.2f}', ha='center', fontsize=8)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


correct_idx = int(np.where(all_preds == all_labels)[0][0])
wrong_idx   = int(np.where(all_preds != all_labels)[0][0])

print('=== Correct prediction ===')
gradcam_plot(test_ds, correct_idx, XAI_DIR / 'gradcam_correct.png')
print('=== Misclassified sample ===')
gradcam_plot(test_ds, wrong_idx,   XAI_DIR / 'gradcam_wrong.png')

---
## 4. Per-Grade Grad-CAM Gallery

For each DR grade, the 3 most confident correct predictions with Grad-CAM overlays.
Shows what retinal features the model associates with each severity level — the primary report figure.

In [ ]:
N = 3
fig = plt.figure(figsize=(N * 4, 5 * 3.5))
fig.suptitle('Grad-CAM++ Gallery — Top Confident Correct Predictions per Grade',
             fontsize=13, fontweight='bold', y=1.01)
outer = gridspec.GridSpec(5, 1, figure=fig, hspace=0.5)

for grade in range(5):
    mask    = (all_labels == grade) & (all_preds == grade)
    indices = np.where(mask)[0]
    if len(indices) == 0:
        print(f'Grade {grade}: no correct predictions, skipping.')
        continue
    top_idx = indices[np.argsort(all_probs[indices, grade])[::-1][:N]]
    inner   = gridspec.GridSpecFromSubplotSpec(1, N, subplot_spec=outer[grade], wspace=0.05)

    for col, idx in enumerate(top_idx):
        ax        = fig.add_subplot(inner[col])
        tensor, _ = get_tensor(test_ds, int(idx))
        rgb_np    = to_np(get_pil(test_ds, int(idx))[0])
        heatmap   = cam(input_tensor=tensor,
                        targets=[ClassifierOutputTarget(grade)])[0]
        overlay   = show_cam_on_image(rgb_np, heatmap, use_rgb=True)
        ax.imshow(overlay); ax.axis('off')
        ax.set_title(f'Conf: {all_probs[idx, grade]*100:.1f}%', fontsize=9)
        if col == 0:
            ax.set_ylabel(f'Grade {grade} — {CLASS_NAMES[grade]}',
                          fontsize=10, fontweight='bold', color=DR_COLORS[grade])

plt.tight_layout()
fig.savefig(XAI_DIR / 'gradcam_gallery.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {XAI_DIR / "gradcam_gallery.png"}')

---
## 5. Attention Rollout — Native SwinV2 Attention

Aggregates self-attention weights across all transformer blocks via the rollout formula (add identity residual → renormalise → multiply across layers). No gradients needed — uses the model's own attention mechanism.

In [ ]:
import cv2


def attention_rollout(input_tensor):
    """Hook attn_drop in every WindowAttention block, apply rollout.
    Returns float32 numpy (224, 224) normalised to [0,1].
    """
    captured = {}
    hooks    = []

    def make_hook(name):
        def hook(module, inp, out):
            captured[name] = out.detach().cpu()   # (B*W, heads, N, N)
        return hook

    for i, stage in enumerate(model.layers):
        for j, blk in enumerate(stage.blocks):
            hooks.append(
                blk.attn.attn_drop.register_forward_hook(make_hook(f's{i}b{j:03d}'))
            )

    model.eval()
    with torch.no_grad():
        _ = model(input_tensor)
    for h in hooks:
        h.remove()

    rollout = None
    for name in sorted(captured.keys()):
        attn = captured[name].mean(dim=0).mean(dim=0)       # (N, N) mean over windows + heads
        attn = attn + torch.eye(attn.shape[-1])              # add residual connection
        attn = attn / attn.sum(dim=-1, keepdim=True)         # renormalise
        if rollout is None:
            rollout = attn
        else:
            rollout = attn @ rollout if rollout.shape == attn.shape else attn

    mask = rollout.mean(dim=0).numpy()
    n    = int(np.sqrt(mask.shape[0]))
    mask = mask[:n*n].reshape(n, n)
    mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
    return cv2.resize(mask, (224, 224))


def rollout_plot(ds, idx, save_path=None):
    tensor, label = get_tensor(ds, idx)
    rgb_np        = to_np(get_pil(ds, idx)[0])
    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1).cpu().numpy()[0]
    pred = int(probs.argmax())
    amap = attention_rollout(tensor)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(
        f'Attention Rollout  |  True: {CLASS_NAMES[label]}  Pred: {CLASS_NAMES[pred]}',
        fontsize=12, fontweight='bold',
        color='green' if pred == label else 'red'
    )
    axes[0].imshow(rgb_np);                           axes[0].set_title('Original');        axes[0].axis('off')
    axes[1].imshow(rgb_np)
    axes[1].imshow(amap, alpha=0.6, cmap='inferno');  axes[1].set_title('Rollout Overlay'); axes[1].axis('off')
    im = axes[2].imshow(amap, cmap='inferno');        axes[2].set_title('Attention Map');   axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# Demo on a Grade-2 (Moderate) correct prediction
g2_idx     = np.where(all_labels == 2)[0]
g2_correct = g2_idx[all_preds[g2_idx] == 2]
demo_idx   = int(g2_correct[0]) if len(g2_correct) else int(g2_idx[0])
rollout_plot(test_ds, demo_idx, XAI_DIR / 'attention_rollout_grade2.png')

---
## 6. Grade 1 False Negatives — Clinically Critical

Grade 1 (Mild DR) images predicted as Grade 0 (No DR) are the most dangerous clinical error — missed early-stage disease that could have been treated.

Grad-CAM reveals why: the model typically attends to the optic disc and major vessels rather than the subtle peripheral microaneurysms that define Grade 1.

In [ ]:
fn_mask = (all_labels == 1) & (all_preds == 0)   # true Grade 1, predicted Grade 0
fn_idx  = np.where(fn_mask)[0]
print(f'Grade 1 -> Grade 0 false negatives: {len(fn_idx)}')

if len(fn_idx) == 0:
    print('No Grade 1 false negatives found — model handles Mild DR well.')
else:
    # Most confident wrong predictions first
    conf_wrong  = all_probs[fn_idx, 0]
    worst_order = fn_idx[np.argsort(conf_wrong)[::-1][:8]]
    n_show      = min(8, len(worst_order))
    cols        = 4
    rows        = (n_show + cols - 1) // cols

    fig, axes = plt.subplots(rows * 2, cols, figsize=(cols * 4, rows * 7))
    if axes.ndim == 1:
        axes = axes.reshape(2, cols)
    axes = axes.reshape(rows * 2, cols)

    fig.suptitle(
        'Grade 1 -> Grade 0 False Negatives (Most Confident Errors)\n'
        'Top row: original  |  Bottom row: Grad-CAM++ explaining the wrong prediction',
        fontsize=11, fontweight='bold'
    )

    for i, idx in enumerate(worst_order[:n_show]):
        row_base = (i // cols) * 2
        col      = i % cols
        tensor, label = get_tensor(test_ds, int(idx))
        rgb_np        = to_np(get_pil(test_ds, int(idx))[0])
        heatmap       = cam(input_tensor=tensor,
                            targets=[ClassifierOutputTarget(0)])[0]   # explain Grade 0 prediction
        overlay       = show_cam_on_image(rgb_np, heatmap, use_rgb=True)
        conf_0        = all_probs[idx, 0]

        axes[row_base,     col].imshow(rgb_np)
        axes[row_base,     col].set_title(f'True: Mild  Pred: No DR\nConf: {conf_0*100:.1f}%',
                                          fontsize=9, color='red')
        axes[row_base,     col].axis('off')
        axes[row_base + 1, col].imshow(overlay)
        axes[row_base + 1, col].set_title('Grad-CAM++ (predicting No DR)', fontsize=8)
        axes[row_base + 1, col].axis('off')

    for i in range(n_show, rows * cols):
        row_base = (i // cols) * 2
        col      = i % cols
        axes[row_base,     col].axis('off')
        axes[row_base + 1, col].axis('off')

    plt.tight_layout()
    fig.savefig(XAI_DIR / 'grade1_false_negatives.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved -> {XAI_DIR / "grade1_false_negatives.png"}')
    print()
    print('Observation: the model attends to the optic disc and major vessel structure.')
    print('Microaneurysms (the defining feature of Grade 1) are subtle and peripheral')
    print('and are not sufficiently captured at the current confidence threshold.')

---
## 7. LIME — Superpixel Explanations

LIME perturbs superpixel regions and measures how each region affects the prediction. Model-agnostic — no access to gradients. Good for verifying Grad-CAM findings independently.

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries
from PIL import Image as PILImage

lime_explainer = lime_image.LimeImageExplainer(random_state=42)


def lime_predict_fn(images_np):
    """LIME passes batches of perturbed uint8 numpy (N, H, W, 3).
    Apply val_transform and return class probabilities (N, 5).
    """
    batch = torch.stack(
        [val_transform(PILImage.fromarray(img)) for img in images_np]
    ).to(device)
    model.eval()
    with torch.no_grad():
        return F.softmax(model(batch), dim=1).cpu().numpy()


def lime_plot(ds, idx, num_samples=500, num_features=8, save_path=None):
    pil, label = get_pil(ds, idx)
    img_np     = np.array(pil.resize((224, 224)))    # uint8 (224, 224, 3)

    with torch.no_grad():
        tensor, _ = get_tensor(ds, idx)
        probs     = F.softmax(model(tensor), dim=1).cpu().numpy()[0]
    pred = int(probs.argmax())

    print(f'  Running LIME ({num_samples} samples)'
          f'  true={CLASS_NAMES[label]}  pred={CLASS_NAMES[pred]} ...')
    explanation = lime_explainer.explain_instance(
        img_np, lime_predict_fn,
        top_labels=5, hide_color=0,
        num_samples=num_samples, random_seed=42
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(
        f'LIME  |  True: {CLASS_NAMES[label]}  '
        f'Pred: {CLASS_NAMES[pred]}  Conf: {probs[pred]*100:.1f}%',
        fontsize=12, fontweight='bold',
        color='green' if pred == label else 'red'
    )
    axes[0].imshow(img_np);  axes[0].set_title('Original');  axes[0].axis('off')

    pos, mask = explanation.get_image_and_mask(
        pred, positive_only=True, num_features=num_features, hide_rest=False)
    axes[1].imshow(mark_boundaries(pos / 255.0, mask))
    axes[1].set_title('Supporting regions');  axes[1].axis('off')

    both, mask2 = explanation.get_image_and_mask(
        pred, positive_only=False, num_features=num_features, hide_rest=False)
    axes[2].imshow(mark_boundaries(both / 255.0, mask2))
    axes[2].set_title('All regions (green=support, red=oppose)');  axes[2].axis('off')

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


lime_plot(test_ds, correct_idx, save_path=XAI_DIR / 'lime_correct.png')
lime_plot(test_ds, wrong_idx,   save_path=XAI_DIR / 'lime_wrong.png')

---
## 8. Misclassification Analysis

In [ ]:
errors     = all_preds.astype(int) - all_labels.astype(int)
abs_errors = np.abs(errors)

# ── 8a. Error distribution + row-normalised confusion matrix ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(errors, bins=np.arange(-4.5, 5.5),
             color='#3498db', edgecolor='white', rwidth=0.8)
axes[0].axvline(0, color='red', linestyle='--', lw=1.5, label='Correct')
axes[0].set_xlabel('Prediction Error (Predicted - True)')
axes[0].set_ylabel('Count')
axes[0].set_title('Signed Error Distribution', fontweight='bold')
axes[0].legend()

cm     = confusion_matrix(all_labels, all_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Purples',
            xticklabels=[f'Pred G{i}' for i in range(5)],
            yticklabels=[f'True G{i}' for i in range(5)],
            ax=axes[1], cbar_kws={'label': '%'})
axes[1].set_title('Confusion Matrix (row-normalised %)', fontweight='bold')
axes[1].set_ylabel('True Grade'); axes[1].set_xlabel('Predicted Grade')

plt.suptitle('Misclassification Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(XAI_DIR / 'misclassification_overview.png', dpi=150, bbox_inches='tight')
plt.show()

for desc, count in [
    ('Correct        ', (abs_errors == 0).sum()),
    ('Off by 1 grade ', (abs_errors == 1).sum()),
    ('Off by >= 2    ', (abs_errors >= 2).sum()),
]:
    print(f'{desc}: {count:5d}  ({count/len(all_labels)*100:.1f}%)')
print(f'\nQWK      : {cohen_kappa_score(all_labels, all_preds, weights="quadratic"):.4f}')
print(f'Macro-F1 : {f1_score(all_labels, all_preds, average="macro"):.4f}')

In [ ]:
# ── 8b. Worst severe misclassifications with Grad-CAM ────────────────────────
severe_idx = np.where(abs_errors >= 2)[0]
print(f'Samples with |error| >= 2: {len(severe_idx)}')

if len(severe_idx) >= 1:
    worst_conf  = all_probs[severe_idx, all_preds[severe_idx]]
    worst_order = severe_idx[np.argsort(worst_conf)[::-1][:4]]

    n_cols = min(4, len(worst_order))
    fig, axes = plt.subplots(2, n_cols, figsize=(n_cols * 4.5, 9))
    if axes.ndim == 1:
        axes = axes.reshape(2, 1)
    fig.suptitle('Severe Misclassifications (|offset| >= 2) — Grad-CAM++ Analysis',
                 fontsize=12, fontweight='bold')

    for col, idx in enumerate(worst_order[:n_cols]):
        tensor, label = get_tensor(test_ds, int(idx))
        rgb_np        = to_np(get_pil(test_ds, int(idx))[0])
        pred          = int(all_preds[idx])
        conf          = all_probs[idx, pred]
        heatmap       = cam(input_tensor=tensor,
                            targets=[ClassifierOutputTarget(pred)])[0]
        overlay       = show_cam_on_image(rgb_np, heatmap, use_rgb=True)

        axes[0, col].imshow(rgb_np);  axes[0, col].axis('off')
        axes[0, col].set_title(
            f'True: {CLASS_NAMES[label]}\nPred: {CLASS_NAMES[pred]}  ({conf*100:.1f}%)',
            fontsize=9, color='red')
        axes[1, col].imshow(overlay); axes[1, col].axis('off')
        axes[1, col].set_title('Grad-CAM++', fontsize=8)

    plt.tight_layout()
    fig.savefig(XAI_DIR / 'worst_misclassifications.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 8c. Per-class Precision, Recall, F1 bar chart ────────────────────────────
f1_per   = f1_score(all_labels,        all_preds, average=None, labels=list(range(5)))
prec_per = precision_score(all_labels, all_preds, average=None, labels=list(range(5)), zero_division=0)
rec_per  = recall_score(all_labels,    all_preds, average=None, labels=list(range(5)), zero_division=0)

x = np.arange(5)
w = 0.22
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x - w,   prec_per, w, label='Precision', color='#3498db', edgecolor='white')
ax.bar(x,       rec_per,  w, label='Recall',    color='#e67e22', edgecolor='white')
ax.bar(x + w,   f1_per,   w, label='F1',        color='#2ecc71', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels([f'G{i} {CLASS_NAMES[i]}' for i in range(5)])
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('Per-Class Precision, Recall, F1 — Test Set', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for i, (p, r, f) in enumerate(zip(prec_per, rec_per, f1_per)):
    ax.text(i - w,   p + 0.02, f'{p:.2f}', ha='center', fontsize=7)
    ax.text(i,       r + 0.02, f'{r:.2f}', ha='center', fontsize=7)
    ax.text(i + w,   f + 0.02, f'{f:.2f}', ha='center', fontsize=7)
plt.tight_layout()
fig.savefig(XAI_DIR / 'per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Confidence Calibration

A well-calibrated model should be correct ~80% of the time when it says 80% confident.
FocalLoss can distort calibration — this is critical to verify for a medical screening model.

In [ ]:
# ── 9a. Reliability diagrams (one-vs-rest per grade) ─────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(17, 4), sharey=True)
fig.suptitle('Reliability Diagrams — Confidence Calibration per DR Grade',
             fontsize=12, fontweight='bold')

for grade in range(5):
    y_bin  = (all_labels == grade).astype(int)
    y_prob = all_probs[:, grade]
    try:
        frac, mean_p = calibration_curve(y_bin, y_prob, n_bins=10, strategy='quantile')
    except ValueError:
        frac, mean_p = np.array([0, 1]), np.array([0, 1])

    ece = sum(
        m.sum() / len(y_prob) * abs(y_bin[m].mean() - y_prob[m].mean())
        for lo, hi in zip(np.linspace(0, 1, 11)[:-1], np.linspace(0, 1, 11)[1:])
        if (m := (y_prob >= lo) & (y_prob < hi)).sum() > 0
    )

    ax = axes[grade]
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect')
    ax.plot(mean_p, frac, 'o-', color=DR_COLORS[grade], lw=2, ms=5, label='Model')
    ax.fill_between(mean_p, frac, mean_p, alpha=0.12, color=DR_COLORS[grade])
    ax.set_title(f'Grade {grade}\nECE={ece:.3f}', fontsize=10,
                 fontweight='bold', color=DR_COLORS[grade])
    ax.set_xlabel('Mean Predicted Prob.')
    if grade == 0:
        ax.set_ylabel('Fraction Positive')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(XAI_DIR / 'calibration_reliability.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 9b. Confidence vs Accuracy tradeoff ──────────────────────────────────────
top_conf = all_probs.max(axis=1)
correct  = (all_preds == all_labels).astype(int)
thresh   = np.linspace(0.0, 0.95, 40)
accs, covs = [], []

for t in thresh:
    m = top_conf >= t
    if m.sum() == 0:
        break
    accs.append(correct[m].mean())
    covs.append(m.mean())

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.plot(thresh[:len(accs)], accs, 'b-o',  ms=4, lw=2, label='Accuracy above threshold')
ax2.plot(thresh[:len(accs)], covs, 'r--s', ms=4, lw=2, label='Coverage (fraction retained)')
ax1.set_xlabel('Confidence Threshold')
ax1.set_ylabel('Accuracy', color='blue')
ax2.set_ylabel('Coverage', color='red')
ax1.set_ylim(0, 1); ax2.set_ylim(0, 1)
ax1.set_title('Confidence vs Accuracy Tradeoff', fontweight='bold')
lines1, lbl1 = ax1.get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lbl1 + lbl2)
ax1.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(XAI_DIR / 'confidence_accuracy_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Summary

In [ ]:
saved = sorted(XAI_DIR.glob('*.png'))
print(f"{'='*62}")
print(f' Explainability outputs -> {XAI_DIR}')
print(f"{'='*62}")
for f in saved:
    print(f'  {f.name:<52s}  {f.stat().st_size/1024:6.1f} KB')

print(f"\n{'─'*62}")
print(' Final test-set metrics')
print(f"{'─'*62}")
print(f'  QWK (primary)   : {cohen_kappa_score(all_labels, all_preds, weights="quadratic"):.4f}')
print(f'  Accuracy        : {accuracy_score(all_labels, all_preds):.4f}')
print(f'  Macro Precision : {precision_score(all_labels, all_preds, average="macro", zero_division=0):.4f}')
print(f'  Macro Recall    : {recall_score(all_labels, all_preds, average="macro", zero_division=0):.4f}')
print(f'  Macro F1        : {f1_score(all_labels, all_preds, average="macro"):.4f}')
print(f'  Severe errors   : {(abs_errors >= 2).sum()} ({(abs_errors >= 2).mean()*100:.1f}%)')
print(f"{'─'*62}")
print()
print(f'  {"Class":<18} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print(f'  {"-"*48}')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:<18} {prec_per[i]:>10.4f} {rec_per[i]:>8.4f} {f1_per[i]:>8.4f}')